In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from datetime import datetime

# Input Model

@dataclass(frozen=True)
class IngestedTelemetryRecord:
    """Ingested record."""
    timestamp: datetime
    rack_temps: List[float]
    airflow_proxy: float
    chiller_power: float
    facility_power: float
    ups_load: float
    tariff: float
    raw_format: str

# Output Model

@dataclass
class ValidatedTelemetryRecord:
    """Validated record with safety metadata."""
    data: IngestedTelemetryRecord
    is_valid: bool = True
    validation_flags: List[str] = field(default_factory=list)
    reason_codes: List[str] = field(default_factory=list)

class EnerviaValidation:
    """Data Validation Service."""

    # Thresholds 
    MAX_RACK_INLET_TEMP = 45.0  # °C 
    SPIKE_THRESHOLD_POWER = 500.0  # kW
    
    def __init__(self, previous_record: Optional[IngestedTelemetryRecord] = None):
        """Initialize with optional previous record."""
        self.previous_record = previous_record

    def validate_record(self, record: IngestedTelemetryRecord) -> ValidatedTelemetryRecord:
        """Validate record constraints."""
        validated = ValidatedTelemetryRecord(data=record)

        # Range checks 
        self._check_temperature_limits(validated)
        self._check_negative_power(validated)

        # Freshness checks 
        self._check_freshness(validated)

        # Spike checks 
        if self.previous_record:
            self._check_spikes(validated)

        # Negative power check
        if validated.data.chiller_power < 0 or validated.data.facility_power < 0:
            validated.is_valid = False
            validated.validation_flags.append("NEGATIVE_POWER_DETECTED")
            validated.reason_codes.append("Power readings cannot be strictly negative.")
            
        return validated

    def _check_temperature_limits(self, v_record: ValidatedTelemetryRecord):
        """Check temp limits."""
        for temp in v_record.data.rack_temps:
            if temp > self.MAX_RACK_INLET_TEMP:
                v_record.is_valid = False
                v_record.validation_flags.append("CRITICAL_TEMP_EXCEEDED")
                v_record.reason_codes.append(f"TEMP_VALUE_{temp}_GT_{self.MAX_RACK_INLET_TEMP}")

    def _check_negative_power(self, v_record: ValidatedTelemetryRecord):
        """Check for negative power."""
        power_metrics = {
            "chiller": v_record.data.chiller_power,
            "facility": v_record.data.facility_power,
            "ups": v_record.data.ups_load
        }
        
        for source, value in power_metrics.items():
            if value < 0:
                v_record.is_valid = False
                v_record.validation_flags.append(f"NEGATIVE_POWER_{source.upper()}")
                v_record.reason_codes.append(f"{source}_power_cannot_be_negative")

    def _check_freshness(self, v_record: ValidatedTelemetryRecord):
        """Check data freshness."""
        if v_record.data.timestamp is None:
            v_record.is_valid = False
            v_record.validation_flags.append("MISSING_TIMESTAMP")

        if self.previous_record:
            # Check for frozen sensors 
            if (v_record.data.chiller_power == self.previous_record.chiller_power and 
                v_record.data.facility_power == self.previous_record.facility_power):
                v_record.validation_flags.append("STALE_DATA_WARNING")

    def _check_spikes(self, v_record: ValidatedTelemetryRecord):
        """Check for spikes."""
        diff = abs(v_record.data.facility_power - self.previous_record.facility_power)
        if diff > self.SPIKE_THRESHOLD_POWER:
            # Flag extreme spikes
            v_record.validation_flags.append("UNREALISTIC_SPIKE_DETECTED")
            v_record.reason_codes.append(f"POWER_JUMP_{diff}_EXCEEDS_THRESHOLD")


# validator = EnerviaValidation(previous_record=last_known_safe_state)
# result = validator.validate_record(current_ingested_record)